In [88]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "hanus2011chimpanzee")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Data_Bottles_opening_pre_test.csv")
complete_path_2 = os.path.join(original_data_pathway, "Data_Bottles_opening_test.csv")
complete_path_3 = os.path.join(original_data_pathway, "Data_Bottles_exchange_pre_test.csv")
complete_path_4 = os.path.join(original_data_pathway, "Data_Bottles_exchange_test.csv")


out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [89]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)
df3 = pd.read_csv(complete_path_3)
df4 = pd.read_csv(complete_path_4)
experiment_import = [[df1, '1', 'opening_pretest'],
                    [df2, '1', 'opening_test'],
                    [df3, '2', 'exchange_pretest'],
                    [df4, '2', 'exchange_test']]
for x, y, k in experiment_import:
    x['experiment']=y
    x['experiment_name']=k





In [90]:
data_frames=[df1, df2, df3, df4]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subject": "participant"}, inplace=True)
    x['participant'] = x['participant'].str.rstrip()
    x['study_id']="hanus2011chimpanzee"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

In [91]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['participant'] = fulldf['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')



In [92]:
datedf=[]
for index, row in fulldf.iterrows():
    if "/" in str(row['date']):
        month,day,year = str(row['date']).split('/')
        datedf.append([month,day,year])
    else:
        try:
            day,month,year = str(row['date']).split('-')
            datedf.append([month,day,year])
        except:
            # print(str(row['date']))
            datedf.append(["","",""])
datedf = pd.DataFrame(datedf, columns=['month', 'day', 'year'])
fulldf = pd.concat([fulldf, datedf], axis=1)

fulldf['month'].replace('oct', '10', inplace=True, regex=True)
fulldf['month'].replace('nov', '11', inplace=True, regex=True)


In [93]:

import re
replace_1=re.compile('(\ |\:|\?)') # / and :
fulldf.columns = fulldf.columns.str.replace(replace_1, '_')
# fulldf.columns = fulldf.columns.str.replace('...', '_')
fulldf.columns = fulldf.columns.str.replace('__', '')
fulldf.rename(columns={"target_given_at_...": "target_given_at"}, inplace=True)
fulldf.dropna(subset=['participant'], inplace=True)
# fulldf.columns

In [94]:
output_no_1 = 1
repeat_1 = 0
init_var = 0
output_no_2 = 1
repeat_2 = 0
temp = []
for index, row in fulldf.iterrows():
    if row['experiment_name'] == 'exchange_test':
        temp.append(output_no_2)
        repeat_2 = repeat_2+1
        if repeat_2 == 4:
            output_no_2 = output_no_2+1
            repeat_2 = 0
        if output_no_2 == 11:
            output_no_2 = 1
    elif row['experiment_name'] == 'opening_test':
        if init_var == 0: ##this only happens once, since init_var will never be 0 again
            init_var = 1 ##it is now 1 
            day_no = row['day'] ## pick up first instance of unique session day
            temp.append(output_no_1) ## append first session number (1)
            continue ## go to the next loop iteration immediately 
        if row['day'] == day_no: ## check to see if day still matches
            temp.append(output_no_1)
        else: ##do not need another if statement
            day_no = row['day'] ##update day_no with changed day
            output_no_1 = output_no_1+1
            if output_no_1 == 9: ##through 8 sessions
                output_no_1 = 1
            temp.append(output_no_1)
    else:
        temp.append(np.nan)
fulldf = fulldf.assign(session=temp)

In [95]:
fulldf['order'].replace(' ', '', inplace=True, regex=True)
split_list = [['first_item_returned', 0, 1], 
                ['second_item_returned', 1, 2],
                ['third_item_returned', 2,3],
                ['fourth_item_returned', 3,4],
                ['fifth_item_returned', 4,5]]
for x, y, k in split_list:
    fulldf[x] = fulldf['order'].str.slice(y,k)


In [96]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])
fulldf['dob'] = pd.to_datetime(fulldf['dob'])

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

fulldf.rename(columns={"age": "age_original",
                       "group":"group_original"}, inplace=True)

fulldf['notes'].replace(' ', '_', inplace=True, regex=True)

In [97]:
fulldf['trial']=fulldf['trial'].astype(int)
fulldf.rename(columns={"trial": "trial_temp"}, inplace=True)

temp = []
for index, row in fulldf.iterrows():
    if row['experiment_name'] == 'exchange_test':
        if row['trial_temp'] > 40:
            temp.append(row['trial_temp']-40)
        else:
            temp.append(row['trial_temp'])
    elif row['experiment_name'] == 'opening_test':
        if row['trial_temp'] > 15:
            temp.append(row['trial_temp']-15)
        else:
            temp.append(row['trial_temp'])
    else:
        temp.append(row['trial_temp'])
fulldf = fulldf.assign(trial=temp)

In [98]:
fulldf=fulldf[['study_id','experiment', 'experiment_name','year', 'month','day',
        'participant', 'age_original','age_in_years','sex','species', 
        'session', 'trial', 'cond', 'juice_side', 'indicated', 'first_opened',
        'group_original',
        'block', 'juice_position',
       'juice_opened_at',   'target_position',
       'target_given_at', 'order', 'first_item_returned', 'second_item_returned', 
       'third_item_returned','fourth_item_returned', 'fifth_item_returned']]


for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'hanus2011chimpanzee_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'hanus2011chimpanzee_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
